### Objetivo de SILVER

Tomar datos crudos (bronze) y producir:

Datos limpios

Métricas financieras fundamentales

Features explícitas para ML

Tablas determinísticas y reproducibles

In [0]:
from pyspark.sql.functions import (
    col, lag, log, exp, when, current_timestamp,
    avg, stddev, max as spark_max, min as spark_min, sum as spark_sum
)
from pyspark.sql.window import Window


In [0]:
#Carga de tablas de bronze
prices = spark.table("workspace.portfolio_intel.bronze_prices")
assets = spark.table("workspace.portfolio_intel.bronze_assets")


## 1 Silver - Precios limpios:


In [0]:
#Usamos adj_close, quitamos nulls, filtramos solo activos activos, normalizamos schema
prices_clean = (
    prices
    .join(
        assets.filter(col("is_active") == True),
        on="symbol",
        how="inner"
    )
    .select(
        col("symbol"),
        col("date"),
        col("adj_close").alias("price"),
        col("volume"),
        col("currency")
    )
    .dropna(subset=["price"])
    .withColumn("is_trading_day", col("volume") > 0)
    .withColumn("cleaned_ts", current_timestamp())
)

prices_clean.write.mode("overwrite").saveAsTable(
    "workspace.portfolio_intel.silver_prices_clean"
)


## 2 Calculo de retornos y  volatilidad

In [0]:
#Window por activo
w = Window.partitionBy("symbol").orderBy("date")


In [0]:
#Cálculo financiero core

returns = (
    prices_clean
    .withColumn("prev_price", lag("price").over(w))
    .withColumn(
        "return_1d",
        when(col("prev_price").isNotNull(),
             log(col("price") / col("prev_price"))
        )
    )
    .withColumn("return_5d", spark_sum("return_1d").over(w.rowsBetween(-4, 0)))
    .withColumn("rolling_vol_20", stddev("return_1d").over(w.rowsBetween(-19, 0)))
)


In [0]:
# Drawdown

w_cum = w.rowsBetween(Window.unboundedPreceding, 0)

returns = (
    returns
    .withColumn("cum_return", spark_sum("return_1d").over(w))
    .withColumn("cum_max", spark_max("cum_return").over(w_cum))
    .withColumn("drawdown", col("cum_return") - col("cum_max"))
    .withColumn("computed_ts", current_timestamp())
)


In [0]:
#Guardamos como tabla
returns.select(
    "symbol", "date", "return_1d", "return_5d", "rolling_vol_20", "drawdown", "computed_ts"
).write.mode("overwrite").saveAsTable(
    "workspace.portfolio_intel.silver_returns"
)


### Window functions son el corazón del análisis financiero en Spark 

## 3 Features para machine learning

Queremos un dataset listo para entrenar. Separaremos entre features y target.

In [0]:
features = (
    returns
    .withColumn("rolling_mean_20", avg("return_1d").over(w.rowsBetween(-19, 0)))
    .withColumn("momentum_10", spark_sum("return_1d").over(w.rowsBetween(-9, 0)))
    .withColumn(
        "target_up_1d",
        when(lag("return_1d", -1).over(w) > 0, 1).otherwise(0)
    )
    .withColumn("feature_ts", current_timestamp())
    .dropna()
)


In [0]:
features.select(
    "symbol", "date",
    "return_1d", "return_5d",
    "rolling_mean_20", "rolling_vol_20",
    "momentum_10", "target_up_1d", "feature_ts"
).write.mode("overwrite").saveAsTable(
    "workspace.portfolio_intel.silver_features_ml"
)


In [0]:
%sql
SELECT symbol, COUNT(*) FROM workspace.portfolio_intel.silver_prices_clean GROUP BY symbol;


symbol,COUNT(*)
SPY,503
QQQ,503
GLD,503
BTC-USD,732


In [0]:
%sql
SELECT * FROM workspace.portfolio_intel.silver_returns ORDER BY date DESC LIMIT 10;


symbol,date,return_1d,return_5d,rolling_vol_20,drawdown,computed_ts
BTC-USD,2026-01-09,-0.009313481130618455,-0.013549026603660842,0.011677816252860436,-0.32448796711373096,2026-01-09T19:39:40.993Z
SPY,2026-01-09,0.007614076303267334,0.01685158537011345,0.005963660762149893,0.0,2026-01-09T19:39:40.993Z
QQQ,2026-01-09,0.010708577247011554,0.02262511802070515,0.009116891154176097,-0.01236356548577755,2026-01-09T19:39:40.993Z
GLD,2026-01-09,0.005090662705926916,0.03772008500389828,0.014271689443374917,-0.0075871452498332825,2026-01-09T19:39:40.993Z
BTC-USD,2026-01-08,-0.003081466892755565,0.004668144474564446,0.011424106162318062,-0.3151744859831125,2026-01-09T19:39:40.993Z
QQQ,2026-01-08,-0.005705242284666945,0.009977525397903898,0.008824140940485411,-0.023072142732789125,2026-01-09T19:39:40.993Z
SPY,2026-01-08,-1.0152683713991636E-4,0.011068890705666568,0.0059060843074164975,-0.0033301333598362848,2026-01-09T19:39:40.993Z
GLD,2026-01-08,0.005507322733438559,0.03758796762619695,0.014266626440392147,-0.012677807955760234,2026-01-09T19:39:40.993Z
GLD,2026-01-07,-0.009605943617543307,0.025591645638933985,0.014263361389327617,-0.018185130689198803,2026-01-09T19:39:40.993Z
QQQ,2026-01-07,9.620289011050677E-4,0.007382761926478763,0.008736540860714985,-0.01736690044812217,2026-01-09T19:39:40.993Z


In [0]:
%sql
SELECT target_up_1d, COUNT(*) FROM workspace.portfolio_intel.silver_features_ml GROUP BY target_up_1d;


target_up_1d,COUNT(*)
0,985
1,1248


###Silver es donde convertimos datos de mercado en información financiera.
###Acá aparecen los retornos, el riesgo y las features que luego alimentan modelos y dashboards.